<a href="https://colab.research.google.com/github/moubanidutta05-del/Time-Series-Animation-NDVI-Assignment-Programming-/blob/main/Time_Series_Animation_NDVI(Assignment_Programming).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import ee
import geemap
ee.Authenticate()
ee.Initialize(project = 'moubani-ee')

In [2]:
Map = geemap.Map()
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

In [14]:
India = ee.FeatureCollection("projects/moubani-ee/assets/National_Boundary")

In [15]:
ndvi = ee.ImageCollection('MODIS/006/MOD13A2').select('NDVI');
Map

Map(bottom=2051.0, center=[25.24469595130604, 92.50694690622291], controls=(WidgetControl(options=['position',…

In [16]:
region = ee.Geometry.Polygon(
    [[[67.060547, 4.740675],
     [67.060547, 38.065392],
     [98.964844, 38.065392],
     [98.964844, 4.740675],
     [67.060547, 4.740675]]],
    None,
    False)
#print(region.getInfo())

In [17]:
ndvi_india = ndvi.map(lambda img: img.set('doy',
ee.Date(img.get('system:time_start')).getRelative('day', 'year')))

In [18]:
distinctDOY = ndvi_india.filterDate('2000-02-18', '2002-02-18');

In [19]:
filter = ee.Filter.equals(leftField='doy', rightField='doy');
join = ee.Join.saveAll('doy_matches');
joinCol = ee.ImageCollection(join.apply(distinctDOY, ndvi_india, filter));

In [20]:
comp = joinCol.map(lambda img: ee.ImageCollection.fromImages(img.get('doy_matches')).reduce(ee.Reducer.mean()).set('doy', ee.Number(img.get('doy'))))

In [21]:
visParams = {
  'min': -2000.0,
  'max': 10000.0,
  'palette': [
    'FFFFFF', 'CE7E45', 'DF923D', 'F1B555', 'FCD163', '99B718', '74A901',
    '66A000', '529400', '3E8601', '207401', '056201', '004C00', '023B01',
    '012E01', '011D01', '011301'
  ],
};

In [22]:
rgbVis = comp.map(lambda img: img.visualize(bands=['NDVI_mean'], **visParams).clip(India))

In [25]:
gifParams = {
  'region': region,
  'dimensions': 600,
  'projection': 'EPSG:4326',
  'framesPerSecond': 10,
  'format': 'gif'
};

In [24]:
print(rgbVis.getVideoThumbURL(gifParams));

https://earthengine.googleapis.com/v1/projects/moubani-ee/videoThumbnails/923ba7adb4e8dbff0221be6f6e16d4dd-02521b186bdb45e41053061db95e0451:getPixels


In [26]:
import os

In [28]:
out_dir = '/content/drive/MyDrive/GEE_Exports/EX6-NDVI_TimeSeries'
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

out_gif = os.path.join(out_dir, 'india_ndvi_animation.gif')

geemap.download_ee_video(rgbVis, gifParams, out_gif)

Generating URL...
Please wait ...
The GIF image has been saved to: /content/drive/MyDrive/GEE_Exports/EX6-NDVI_TimeSeries/india_ndvi_animation.gif
